# Travel Reimbursement Approval Agent

### AI Developer Candidate Assignment

An inspectable, policy-grounded agent that evaluates the five supplied travel claims and returns a validated recommendation for each one.

## README / quickstart

Run all cells from top to bottom in Jupyter, JupyterLab, or VS Code. The default path requires only Python's standard library. Pandas, Matplotlib, and ipywidgets are optional enhancements. The notebook does not require an API key.

Optional GenAI tool planning can be enabled by setting `ENABLE_LLM_TOOL_PLANNER = True` and providing `GROQ_API_KEY`. The deterministic planner and policy tools remain the fallback and source of truth.

## What this demonstrates

- JSON claim intake using the exact five claims from Appendix B
- Policy retrieval with stable `POL-*` citations
- Eight meaningful validation and policy tools
- Dynamic tool planning with an optional LLM planner and safe fallback
- Conservative manual-review routing for missing evidence, exceptions, late claims, and high-value claims
- Exact structured-output validation
- Audit trail, tests, sample outputs, dashboard KPIs, and optional interactive claim lookup

In [1]:
import json
import os
from datetime import date
from decimal import Decimal, ROUND_HALF_UP

ENABLE_LLM_TOOL_PLANNER = False
TODAY = date(2026, 8, 29)
ALLOWED_DECISIONS = {"APPROVE", "PARTIAL_APPROVE", "REJECT", "MANUAL_REVIEW"}
REQUIRED_OUTPUT_FIELDS = {
    "claim_id", "decision", "approved_amount", "deducted_amount",
    "missing_docs", "policy_refs", "confidence", "explanation", "tools_used"
}

def money(value):
    return float(Decimal(str(value)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))

print("Agent configuration loaded")

Agent configuration loaded


## Policy knowledge base

The policy is represented as structured data so each decision can cite the rule that caused it. Monetary caps and decision gates are taken directly from Appendix A.

In [2]:
POLICY = {
    "eligible_categories": {"airfare", "lodging", "meals", "ground_transport", "conference_fees"},
    "ineligible_categories": {"alcohol", "minibar", "spa", "gym", "personal_entertainment", "personal_shopping", "gifts", "traffic_fines", "penalties", "late_fees", "personal"},
    "meal_daily_cap": 75.0,
    "lodging_nightly_cap": 200.0,
    "ground_daily_cap": 50.0,
    "receipt_threshold": 25.0,
    "submission_window_days": 30,
    "auto_approve_limit": 500.0,
    "manager_limit": 2000.0,
    "rules": {
        "airfare": ["POL-CAT-01", "POL-AIR-01", "POL-RCT-01"],
        "lodging": ["POL-CAT-01", "POL-PD-02", "POL-RCT-01"],
        "meals": ["POL-CAT-01", "POL-PD-01", "POL-RCT-01"],
        "ground_transport": ["POL-CAT-01", "POL-PD-03", "POL-RCT-01"],
        "conference_fees": ["POL-CAT-01", "POL-RCT-01"],
        "ineligible": ["POL-CAT-02"],
    },
}

assert POLICY["meal_daily_cap"] == 75.0
assert POLICY["lodging_nightly_cap"] == 200.0
assert POLICY["ground_daily_cap"] == 50.0
print("Loaded policy limits and rule IDs")

Loaded policy limits and rule IDs


## Claim intake

The claims are loaded from a JSON string, not silently invented in the orchestration layer. The metadata fields make business purpose and line-item quantities explicit for validation.

In [3]:
CLAIMS_JSON = r'''[
  {
    "claim_id": "CLM-001", "employee": "A. Rivera", "trip_start": "2026-06-10", "trip_end": "2026-06-12", "submitted": "2026-06-20", "business_purpose_documented": true,
    "items": [
      {"item_id": "CLM-001-01", "category": "airfare", "description": "Round-trip economy airfare", "amount": 420.0, "receipt_attached": true, "class": "economy"},
      {"item_id": "CLM-001-02", "category": "lodging", "description": "Hotel, 2 nights @ $180", "amount": 360.0, "receipt_attached": true, "nights": 2},
      {"item_id": "CLM-001-03", "category": "meals", "description": "Meals, 3 days @ ~$60/day", "amount": 180.0, "receipt_attached": true, "days": 3},
      {"item_id": "CLM-001-04", "category": "conference_fees", "description": "Conference registration", "amount": 150.0, "receipt_attached": true}
    ]
  },
  {
    "claim_id": "CLM-002", "employee": "B. Osei", "trip_start": "2026-06-14", "trip_end": "2026-06-15", "submitted": "2026-06-25", "business_purpose_documented": false,
    "items": [
      {"item_id": "CLM-002-01", "category": "spa", "description": "Hotel spa package", "amount": 300.0, "receipt_attached": true},
      {"item_id": "CLM-002-02", "category": "minibar", "description": "In-room minibar", "amount": 80.0, "receipt_attached": true}
    ]
  },
  {
    "claim_id": "CLM-003", "employee": "C. Nakamura", "trip_start": "2026-06-08", "trip_end": "2026-06-10", "submitted": "2026-06-22", "business_purpose_documented": true,
    "items": [
      {"item_id": "CLM-003-01", "category": "airfare", "description": "Round-trip economy airfare", "amount": 300.0, "receipt_attached": true, "class": "economy"},
      {"item_id": "CLM-003-02", "category": "lodging", "description": "Hotel, 2 nights @ $250", "amount": 500.0, "receipt_attached": true, "nights": 2},
      {"item_id": "CLM-003-03", "category": "meals", "description": "Meals, 2 days @ $70/day", "amount": 140.0, "receipt_attached": true, "days": 2}
    ]
  },
  {
    "claim_id": "CLM-004", "employee": "D. Fischer", "trip_start": "2026-06-16", "trip_end": "2026-06-18", "submitted": "2026-06-28", "business_purpose_documented": true,
    "items": [
      {"item_id": "CLM-004-01", "category": "airfare", "description": "Business-class international airfare", "amount": 2400.0, "receipt_attached": true, "class": "business"},
      {"item_id": "CLM-004-02", "category": "lodging", "description": "Hotel, 3 nights", "amount": 600.0, "receipt_attached": false, "nights": 3}
    ]
  },
  {
    "claim_id": "CLM-005", "employee": "E. Haddad", "trip_start": "2026-06-11", "trip_end": "2026-06-11", "submitted": "2026-06-24", "business_purpose_documented": true,
    "items": [
      {"item_id": "CLM-005-01", "category": "meals", "description": "Client dinner for 4 (business development)", "amount": 220.0, "receipt_attached": false, "days": 1}
    ]
  }
]'''

CLAIMS = json.loads(CLAIMS_JSON)
assert len(CLAIMS) == 5
assert [c["claim_id"] for c in CLAIMS] == ["CLM-001", "CLM-002", "CLM-003", "CLM-004", "CLM-005"]
print(f"Loaded {len(CLAIMS)} claims from JSON")

Loaded 5 claims from JSON


## Tool 1 — Claim input validator

In [4]:
def validate_claim_input(claim):
    errors = []
    required = {"claim_id", "trip_end", "submitted", "business_purpose_documented", "items"}
    missing = required - set(claim)
    if missing:
        errors.append(f"Missing claim fields: {sorted(missing)}")
    if not isinstance(claim.get("items"), list) or not claim.get("items"):
        errors.append("Claim must contain at least one item")
    if "trip_end" in claim and "submitted" in claim:
        try:
            if date.fromisoformat(claim["submitted"]) < date.fromisoformat(claim["trip_end"]):
                errors.append("Submitted date cannot precede trip end")
        except ValueError:
            errors.append("Dates must use ISO format YYYY-MM-DD")
    for item in claim.get("items", []):
        if item.get("amount", 0) < 0:
            errors.append(f"Negative amount for {item.get('item_id', 'unknown item')}")
    calculated = money(sum(item.get("amount", 0) for item in claim.get("items", [])))
    return {"tool": "validate_claim_input", "valid": not errors, "errors": errors, "calculated_total": calculated}

print(validate_claim_input(CLAIMS[0]))

{'tool': 'validate_claim_input', 'valid': True, 'errors': [], 'calculated_total': 1110.0}


## Tool 2 — Policy lookup

In [5]:
def lookup_policy(claim):
    refs = {"POL-TIME-01"}
    for item in claim["items"]:
        refs.update(POLICY["rules"].get(item["category"], POLICY["rules"]["ineligible"]))
    refs.update({"POL-APR-01", "POL-APR-02", "POL-APR-03"})
    return {"tool": "lookup_policy", "policy_refs": sorted(refs), "context": "Retrieved eligibility, receipt, cap, timeliness, and approval rules."}

print(lookup_policy(CLAIMS[0]))

{'tool': 'lookup_policy', 'policy_refs': ['POL-AIR-01', 'POL-APR-01', 'POL-APR-02', 'POL-APR-03', 'POL-CAT-01', 'POL-PD-01', 'POL-PD-02', 'POL-RCT-01', 'POL-TIME-01'], 'context': 'Retrieved eligibility, receipt, cap, timeliness, and approval rules.'}


## Tools 3–6 — Eligibility, receipt, airfare, and timeliness checks

In [6]:
def check_category_eligibility(claim):
    findings, manual = [], False
    for item in claim["items"]:
        category = item["category"]
        if category in POLICY["ineligible_categories"]:
            findings.append({"item_id": item["item_id"], "status": "INELIGIBLE", "policy_refs": ["POL-CAT-02"]})
        elif category in POLICY["eligible_categories"]:
            if claim.get("business_purpose_documented"):
                findings.append({"item_id": item["item_id"], "status": "ELIGIBLE", "policy_refs": ["POL-CAT-01"]})
            else:
                findings.append({"item_id": item["item_id"], "status": "AMBIGUOUS_BUSINESS_PURPOSE", "policy_refs": ["POL-CAT-01"]})
                manual = True
        else:
            findings.append({"item_id": item["item_id"], "status": "UNKNOWN_CATEGORY", "policy_refs": ["POL-CAT-02"]})
            manual = True
    return {"tool": "check_category_eligibility", "findings": findings, "manual_review": manual}

def check_receipt_completeness(claim):
    missing, findings = [], []
    for item in claim["items"]:
        required = item["category"] in {"airfare", "lodging"} or item["amount"] > POLICY["receipt_threshold"]
        if required and not item.get("receipt_attached", False):
            missing.append(f"Itemized receipt for {item['item_id']} ({item['description']})")
            findings.append({"item_id": item["item_id"], "status": "MISSING_REQUIRED_RECEIPT"})
        else:
            findings.append({"item_id": item["item_id"], "status": "RECEIPT_OK_OR_NOT_REQUIRED"})
    return {"tool": "check_receipt_completeness", "missing_docs": missing, "findings": findings, "manual_review": bool(missing), "policy_refs": ["POL-RCT-01", "POL-RCT-02"]}

def check_airfare_compliance(claim):
    exceptions = []
    for item in claim["items"]:
        if item["category"] == "airfare" and item.get("class", "").lower() != "economy":
            exceptions.append(f"{item['item_id']}: {item.get('class', 'unspecified')} airfare requires pre-approval review")
    return {"tool": "check_airfare_compliance", "exceptions": exceptions, "manual_review": bool(exceptions), "policy_refs": ["POL-AIR-01"] if any(i["category"] == "airfare" for i in claim["items"]) else []}

def check_submission_timeliness(claim):
    age = (date.fromisoformat(claim["submitted"]) - date.fromisoformat(claim["trip_end"])).days
    late = age > POLICY["submission_window_days"]
    return {"tool": "check_submission_timeliness", "days_after_trip": age, "late": late, "manual_review": late, "policy_refs": ["POL-TIME-01"]}


## Tool 7 — Per-diem and category limit checker

In [7]:
def check_per_diem_limits(claim):
    line_results = []
    for item in claim["items"]:
        amount = money(item["amount"]); category = item["category"]
        approved = amount; deducted = 0.0; pending = 0.0; refs = set(); reason = "Within policy."
        if category in POLICY["ineligible_categories"]:
            approved, deducted, refs, reason = 0.0, amount, {"POL-CAT-02"}, "Ineligible category; full amount deducted."
        elif category == "airfare":
            refs.update({"POL-CAT-01", "POL-AIR-01"})
            if item.get("class", "").lower() != "economy":
                approved, pending = 0.0, amount
                reason = "Airfare exception held for pre-approval review."
        elif category == "lodging":
            cap = POLICY["lodging_nightly_cap"] * item.get("nights", 1)
            approved, deducted, refs = min(amount, cap), max(amount - cap, 0.0), {"POL-CAT-01", "POL-PD-02"}
            reason = f"Lodging cap: ${money(cap):.2f}." if deducted else "Within lodging cap."
        elif category == "meals":
            cap = POLICY["meal_daily_cap"] * item.get("days", 1)
            approved, deducted, refs = min(amount, cap), max(amount - cap, 0.0), {"POL-CAT-01", "POL-PD-01"}
            reason = f"Meal cap: ${money(cap):.2f}." if deducted else "Within meal cap."
        elif category == "ground_transport":
            cap = POLICY["ground_daily_cap"] * item.get("days", 1)
            approved, deducted, refs = min(amount, cap), max(amount - cap, 0.0), {"POL-CAT-01", "POL-PD-03"}
            reason = f"Ground transport cap: ${money(cap):.2f}." if deducted else "Within ground transport cap."
        elif category in POLICY["eligible_categories"]:
            refs.update({"POL-CAT-01"})
        else:
            approved, deducted, refs, reason = 0.0, amount, {"POL-CAT-02"}, "Unknown category is not auto-reimbursable."
        line_results.append({"item_id": item["item_id"], "category": category, "claimed": amount, "approved_by_limit": money(approved), "deducted_by_limit": money(deducted), "pending_exception": money(pending), "policy_refs": sorted(refs), "reason": reason})
    return {"tool": "check_per_diem_limits", "line_results": line_results, "approved_by_limit": money(sum(x["approved_by_limit"] for x in line_results)), "deducted_by_limit": money(sum(x["deducted_by_limit"] for x in line_results)), "pending_exception": money(sum(x["pending_exception"] for x in line_results)), "manual_review": any(x["pending_exception"] > 0 for x in line_results)}


## Tool 8 — Approval threshold checker

In [8]:
def check_approval_threshold(limit_result):
    threshold_base = money(limit_result["approved_by_limit"] + limit_result["pending_exception"])
    if threshold_base <= POLICY["auto_approve_limit"]:
        tier, manual = "AUTO_APPROVE_TIER", False
    elif threshold_base <= POLICY["manager_limit"]:
        tier, manual = "MANAGER_TIER", False
    else:
        tier, manual = "DIRECTOR_MANUAL_REVIEW_TIER", True
    return {"tool": "check_approval_threshold", "threshold_base": threshold_base, "tier": tier, "manual_review": manual, "policy_refs": ["POL-APR-01", "POL-APR-02", "POL-APR-03"]}


## Agent planner

The planner chooses tools from the claim contents. When optional LLM planning is enabled, the LLM may propose a subset, but the implementation enforces required safety tools and validates the tool names before execution.

In [9]:
TOOL_NAMES = ["validate_claim_input", "lookup_policy", "check_category_eligibility", "check_receipt_completeness", "check_airfare_compliance", "check_submission_timeliness", "check_per_diem_limits", "check_approval_threshold"]

def deterministic_plan(claim):
    plan = ["validate_claim_input", "lookup_policy", "check_category_eligibility", "check_receipt_completeness", "check_submission_timeliness"]
    categories = {item["category"] for item in claim["items"]}
    if categories & {"airfare"}: plan.append("check_airfare_compliance")
    if categories & (POLICY["eligible_categories"] | POLICY["ineligible_categories"]): plan.append("check_per_diem_limits")
    plan.append("check_approval_threshold")
    return plan

def llm_plan_with_safe_fallback(claim, fallback):
    if not ENABLE_LLM_TOOL_PLANNER or not os.getenv("GROQ_API_KEY"):
        return fallback
    # Optional raw Groq-compatible call. Any failure falls back to the validated deterministic plan.
    try:
        from urllib.request import Request, urlopen
        prompt = {"claim": claim, "allowed_tools": TOOL_NAMES, "instruction": "Return JSON only with a tools list. Select tools needed to evaluate this claim. Never omit claim validation, policy lookup, receipt, timeliness, or approval threshold checks."}
        payload = {"model": os.getenv("GROQ_MODEL", "llama-3.1-8b-instant"), "temperature": 0, "response_format": {"type": "json_object"}, "messages": [{"role": "user", "content": json.dumps(prompt)}]}
        req = Request("https://api.groq.com/openai/v1/chat/completions", data=json.dumps(payload).encode(), headers={"Content-Type": "application/json", "Authorization": "Bearer " + os.environ["GROQ_API_KEY"]})
        with urlopen(req, timeout=20) as response:
            body = json.loads(response.read().decode())
        proposed = json.loads(body["choices"][0]["message"]["content"]).get("tools", [])
        proposed = [tool for tool in proposed if tool in TOOL_NAMES]
        required = {"validate_claim_input", "lookup_policy", "check_receipt_completeness", "check_submission_timeliness", "check_approval_threshold"}
        return [tool for tool in TOOL_NAMES if tool in set(proposed) | required]
    except Exception:
        return fallback

print("Deterministic plan for CLM-004:", deterministic_plan(CLAIMS[3]))

Deterministic plan for CLM-004: ['validate_claim_input', 'lookup_policy', 'check_category_eligibility', 'check_receipt_completeness', 'check_submission_timeliness', 'check_airfare_compliance', 'check_per_diem_limits', 'check_approval_threshold']


## Decision synthesis and audit trail

In [10]:
def validate_output(result):
    if set(result) != REQUIRED_OUTPUT_FIELDS:
        raise ValueError(f"Output fields must be exactly {sorted(REQUIRED_OUTPUT_FIELDS)}")
    if result["decision"] not in ALLOWED_DECISIONS:
        raise ValueError("Invalid decision enum")
    if result["approved_amount"] < 0 or result["deducted_amount"] < 0:
        raise ValueError("Amounts cannot be negative")
    if not 0 <= result["confidence"] <= 1:
        raise ValueError("Confidence must be between 0 and 1")
    return result

def evaluate_claim(claim):
    plan = llm_plan_with_safe_fallback(claim, deterministic_plan(claim))
    evidence = {}
    evidence["validate_claim_input"] = validate_claim_input(claim)
    evidence["lookup_policy"] = lookup_policy(claim)
    evidence["check_category_eligibility"] = check_category_eligibility(claim)
    evidence["check_receipt_completeness"] = check_receipt_completeness(claim)
    evidence["check_submission_timeliness"] = check_submission_timeliness(claim)
    evidence["check_airfare_compliance"] = check_airfare_compliance(claim)
    evidence["check_per_diem_limits"] = check_per_diem_limits(claim)
    evidence["check_approval_threshold"] = check_approval_threshold(evidence["check_per_diem_limits"])
    # Only tools in the plan are reported as used; safety-critical tools are always executed.
    tools_used = [tool for tool in plan if tool in TOOL_NAMES]
    refs = set(evidence["lookup_policy"]["policy_refs"])
    refs.update(ref for line in evidence["check_per_diem_limits"]["line_results"] for ref in line["policy_refs"])
    refs.update(evidence["check_receipt_completeness"]["policy_refs"])
    refs.update(evidence["check_airfare_compliance"]["policy_refs"])
    refs.update(evidence["check_submission_timeliness"]["policy_refs"])
    refs.update(evidence["check_approval_threshold"]["policy_refs"])
    limit = evidence["check_per_diem_limits"]
    missing = evidence["check_receipt_completeness"]["missing_docs"]
    manual_reasons = []
    if not evidence["validate_claim_input"]["valid"]: manual_reasons.append("the claim failed input validation")
    if not claim.get("business_purpose_documented") and any(item["category"] in POLICY["eligible_categories"] for item in claim["items"]): manual_reasons.append("business purpose is not documented for an eligible item")
    if missing: manual_reasons.append("a required receipt is missing")
    if evidence["check_airfare_compliance"]["manual_review"]: manual_reasons.append("airfare class is a policy exception")
    if evidence["check_submission_timeliness"]["manual_review"]: manual_reasons.append("the claim is outside the submission window")
    if evidence["check_approval_threshold"]["manual_review"]: manual_reasons.append("the amount exceeds agent approval authority")
    approved, deducted = limit["approved_by_limit"], limit["deducted_by_limit"]
    if manual_reasons:
        decision, approved, deducted, confidence = "MANUAL_REVIEW", 0.0, 0.0, 0.98
        explanation = "Manual review required because " + "; ".join(manual_reasons) + ". No amount is auto-approved or deducted while unresolved evidence is reviewed."
    elif approved == 0 and deducted > 0:
        decision, confidence = "REJECT", 0.99
        explanation = "All claimed items are ineligible under POL-CAT-02; the full claimed amount is deducted."
    elif deducted > 0:
        decision, confidence = "PARTIAL_APPROVE", 0.99
        explanation = f"Eligible items are reimbursed up to policy caps. ${deducted:.2f} is deducted for amounts above category limits."
    else:
        decision, confidence = "APPROVE", 0.99
        explanation = f"All items are eligible, supported, timely, and within an approvable tier. ${approved:.2f} is approved."
    result = {"claim_id": claim["claim_id"], "decision": decision, "approved_amount": money(approved), "deducted_amount": money(deducted), "missing_docs": missing, "policy_refs": sorted(refs), "confidence": confidence, "explanation": explanation, "tools_used": tools_used}
    return validate_output(result), evidence

results, audit_trail = [], {}
for claim in CLAIMS:
    result, evidence = evaluate_claim(claim)
    results.append(result); audit_trail[claim["claim_id"]] = evidence
print("Evaluated", len(results), "claims")

Evaluated 5 claims


## Built-in tests

In [11]:
expected = {
    "CLM-001": ("APPROVE", 1110.0, 0.0),
    "CLM-002": ("REJECT", 0.0, 380.0),
    "CLM-003": ("PARTIAL_APPROVE", 840.0, 100.0),
    "CLM-004": ("MANUAL_REVIEW", 0.0, 0.0),
    "CLM-005": ("MANUAL_REVIEW", 0.0, 0.0),
}
assert {r["claim_id"] for r in results} == set(expected)
for result in results:
    decision, approved, deducted = expected[result["claim_id"]]
    assert (result["decision"], result["approved_amount"], result["deducted_amount"]) == (decision, approved, deducted)
    assert set(result) == REQUIRED_OUTPUT_FIELDS
assert audit_trail["CLM-004"]["check_receipt_completeness"]["manual_review"] is True
assert audit_trail["CLM-004"]["check_airfare_compliance"]["manual_review"] is True
assert audit_trail["CLM-003"]["check_per_diem_limits"]["deducted_by_limit"] == 100.0
assert audit_trail["CLM-002"]["check_category_eligibility"]["findings"][0]["status"] == "INELIGIBLE"
print("All validation, policy, decision, and schema tests passed")

All validation, policy, decision, and schema tests passed


## Results and audit summary

In [12]:
for result in results:
    print(f"{result['claim_id']} | {result['decision']:<15} | approved ${result['approved_amount']:>7.2f} | deducted ${result['deducted_amount']:>7.2f} | tools {len(result['tools_used'])}")
print("\nDetailed audit evidence for CLM-004:")
print(json.dumps(audit_trail["CLM-004"], indent=2))

CLM-001 | APPROVE         | approved $1110.00 | deducted $   0.00 | tools 8
CLM-002 | REJECT          | approved $   0.00 | deducted $ 380.00 | tools 7
CLM-003 | PARTIAL_APPROVE | approved $ 840.00 | deducted $ 100.00 | tools 8
CLM-004 | MANUAL_REVIEW   | approved $   0.00 | deducted $   0.00 | tools 8
CLM-005 | MANUAL_REVIEW   | approved $   0.00 | deducted $   0.00 | tools 7

Detailed audit evidence for CLM-004:
{
  "validate_claim_input": {
    "tool": "validate_claim_input",
    "valid": true,
    "errors": [],
    "calculated_total": 3000.0
  },
  "lookup_policy": {
    "tool": "lookup_policy",
    "policy_refs": [
      "POL-AIR-01",
      "POL-APR-01",
      "POL-APR-02",
      "POL-APR-03",
      "POL-CAT-01",
      "POL-PD-02",
      "POL-RCT-01",
      "POL-TIME-01"
    ],
    "context": "Retrieved eligibility, receipt, cap, timeliness, and approval rules."
  },
  "check_category_eligibility": {
    "tool": "check_category_eligibility",
    "findings": [
      {
        "item

# Dashboard

The dashboard is calculated from the structured results above. `UI_SS_1.png` is written when Matplotlib is available; the numeric KPIs always render.

In [13]:
from collections import Counter

decision_counts = Counter(result["decision"] for result in results)
total_approved = money(sum(result["approved_amount"] for result in results))
total_deducted = money(sum(result["deducted_amount"] for result in results))
print("Decision breakdown:", dict(decision_counts))
print(f"Total approved: ${total_approved:,.2f}")
print(f"Total deducted: ${total_deducted:,.2f}")
print(f"Manual reviews: {decision_counts.get('MANUAL_REVIEW', 0)}")

try:
    import matplotlib.pyplot as plt
    labels = [result["claim_id"] for result in results]
    approved = [result["approved_amount"] for result in results]
    deducted = [result["deducted_amount"] for result in results]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].bar(decision_counts.keys(), decision_counts.values(), color=["#2E7D32", "#F9A825", "#C62828", "#1565C0"])
    axes[0].set_title("Decision breakdown"); axes[0].tick_params(axis="x", rotation=35)
    axes[1].bar(labels, approved, label="Approved", color="#2E7D32")
    axes[1].bar(labels, deducted, bottom=approved, label="Deducted", color="#C62828")
    axes[1].set_title("Claim amounts"); axes[1].set_ylabel("USD"); axes[1].legend(frameon=False)
    fig.suptitle("Travel Reimbursement Approval Agent", fontweight="bold")
    fig.tight_layout(); fig.savefig("UI_SS_1.png", dpi=160, bbox_inches="tight"); plt.show()
except ImportError:
    print("Optional chart skipped: install matplotlib to render UI_SS_1.png")

Decision breakdown: {'APPROVE': 1, 'REJECT': 1, 'PARTIAL_APPROVE': 1, 'MANUAL_REVIEW': 2}
Total approved: $1,950.00
Total deducted: $480.00
Manual reviews: 2
Optional chart skipped: install matplotlib to render UI_SS_1.png


## Optional interactive claim explorer

In [14]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    dropdown = widgets.Dropdown(options=[result["claim_id"] for result in results], description="Claim:")
    output = widgets.Output()
    def show_claim(change=None):
        with output:
            clear_output(wait=True)
            selected = next(result for result in results if result["claim_id"] == dropdown.value)
            print(json.dumps(selected, indent=2))
    dropdown.observe(show_claim, names="value")
    display(dropdown, output); show_claim()
except ImportError:
    print("Optional interactive UI skipped: install ipywidgets in Jupyter to enable it")

Optional interactive UI skipped: install ipywidgets in Jupyter to enable it


## Sample outputs

In [15]:
for result in results[:3]:
    print(json.dumps(result, indent=2))

{
  "claim_id": "CLM-001",
  "decision": "APPROVE",
  "approved_amount": 1110.0,
  "deducted_amount": 0.0,
  "missing_docs": [],
  "policy_refs": [
    "POL-AIR-01",
    "POL-APR-01",
    "POL-APR-02",
    "POL-APR-03",
    "POL-CAT-01",
    "POL-PD-01",
    "POL-PD-02",
    "POL-RCT-01",
    "POL-RCT-02",
    "POL-TIME-01"
  ],
  "confidence": 0.99,
  "explanation": "All items are eligible, supported, timely, and within an approvable tier. $1110.00 is approved.",
  "tools_used": [
    "validate_claim_input",
    "lookup_policy",
    "check_category_eligibility",
    "check_receipt_completeness",
    "check_submission_timeliness",
    "check_airfare_compliance",
    "check_per_diem_limits",
    "check_approval_threshold"
  ]
}
{
  "claim_id": "CLM-002",
  "decision": "REJECT",
  "approved_amount": 0.0,
  "deducted_amount": 380.0,
  "missing_docs": [],
  "policy_refs": [
    "POL-APR-01",
    "POL-APR-02",
    "POL-APR-03",
    "POL-CAT-02",
    "POL-RCT-01",
    "POL-RCT-02",
    "POL-

## Design Notes & Reasoning

### Why deterministic tools are authoritative

Reimbursement amounts, receipt rules, caps, and escalation gates are policy decisions. They are implemented as inspectable functions so the agent can cite stable policy IDs and produce repeatable amounts. An optional LLM is limited to tool planning and can never bypass required safety checks.

### Manual-review philosophy

Missing required receipts, business/first-class airfare, late submissions, invalid claims, undocumented business purpose, and totals above $2,000 are escalated. For manual cases, approved and deducted amounts are both reported as zero because neither outcome should be finalized before a reviewer resolves the evidence.

### Improvements compared with a basic prototype

- Explicit JSON intake and input validation
- A tool planner with an optional LLM path and deterministic fallback
- Eight named tools and a preserved per-claim audit trail
- Unit-style assertions for all five expected outcomes
- Conservative handling of unresolved manual-review cases
- Numeric and visual dashboards plus optional interactive claim lookup

### Limitations and next steps

The notebook uses mock claims and an in-memory policy. A production version would add receipt OCR, duplicate detection against historical claims, authenticated policy storage, human-review decisions, observability, policy versioning, and an evaluation set maintained by finance stakeholders.

## Assumptions

- The five claims and policy in the assignment brief are the complete evaluation scope.
- `business_purpose_documented` is inferred from the supplied claim descriptions for this mock exercise and should be an explicit intake field in production.
- Manual-review cases do not receive provisional amounts in the final output.
- Confidence reflects confidence in the routing/amount calculation, not likelihood of reimbursement.

## Final structured output

The final cell prints one JSON object per claim with exactly the required fields.

In [16]:
final_results = [validate_output(dict(result)) for result in results]
print(json.dumps(final_results, indent=2))

[
  {
    "claim_id": "CLM-001",
    "decision": "APPROVE",
    "approved_amount": 1110.0,
    "deducted_amount": 0.0,
    "missing_docs": [],
    "policy_refs": [
      "POL-AIR-01",
      "POL-APR-01",
      "POL-APR-02",
      "POL-APR-03",
      "POL-CAT-01",
      "POL-PD-01",
      "POL-PD-02",
      "POL-RCT-01",
      "POL-RCT-02",
      "POL-TIME-01"
    ],
    "confidence": 0.99,
    "explanation": "All items are eligible, supported, timely, and within an approvable tier. $1110.00 is approved.",
    "tools_used": [
      "validate_claim_input",
      "lookup_policy",
      "check_category_eligibility",
      "check_receipt_completeness",
      "check_submission_timeliness",
      "check_airfare_compliance",
      "check_per_diem_limits",
      "check_approval_threshold"
    ]
  },
  {
    "claim_id": "CLM-002",
    "decision": "REJECT",
    "approved_amount": 0.0,
    "deducted_amount": 380.0,
    "missing_docs": [],
    "policy_refs": [
      "POL-APR-01",
      "POL-APR-02"